<a href="https://colab.research.google.com/github/nmathgithub/QuantML2024/blob/ml_project2024/StockTracker2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Undervalued Stocks using Selenium
# We use Selenium since there are interactive components due to javascript
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time

# Configure the WebDriver
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Run in headless mode
options.add_argument("--no-sandbox") # Bypass OS security model
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=options)

# Open the URL
url = 'https://finance.yahoo.com/research-hub/screener/undervalued_growth_stocks/'
driver.get(url)

# Wait for the page to load
time.sleep(5)

# Locate the table rows
rows = driver.find_elements(By.CSS_SELECTOR, 'table tbody tr')

# Extract data
data = []
for row in rows[:7]:  # Get the top 7 rows
    cols = row.find_elements(By.TAG_NAME, 'td')

    # Extract the symbol and full name
    symbol = cols[1].text.strip()
    name = cols[2].text.strip()

    # Extract only the stock symbol from the full name
    symbol = symbol.split('\n')[1]  # Get the part after the newline

    # Extract other columns
    price = '$' + cols[4].text.strip()
    change = cols[5].text.strip()
    percent_change = cols[6].text.strip()
    market_cap = cols[9].text.strip()

    data.append([symbol, name, price, change, percent_change, market_cap])

# Define column names
columns = ['Symbol', 'Name', 'Price', 'Change', '% Change', 'Market Cap']

# Create a DataFrame
df_undervalued = pd.DataFrame(data, columns=columns)

# Close the driver
driver.quit()

# Display the DataFrame
print(df_undervalued)
undervalued_html = df_undervalued.to_html(index=False)

# ------
#
import requests
from bs4 import BeautifulSoup
import pandas as pd
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import os

# Define the URL and headers
url = 'https://finance.yahoo.com/markets/stocks/gainers/'
headers = {'User-Agent': 'Mozilla/5.0'}

# Send the request and get the HTML content
response = requests.get(url, headers=headers)
html_content = response.text

# Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# Find the table containing the most active stocks
table = soup.find('table')  # This is where the actual stock data is located
if table is None:
    raise ValueError("Table not found. Check the website's structure.")

# Extract table rows
rows = table.find_all('tr')[1:7]  # Skip header row and limit to 6 rows

# Prepare lists to hold the extracted data
data = []

# Iterate over the rows and extract data
for row in rows:
    cols = row.find_all('td')

    # Ensure there are enough columns in the row
    if len(cols) < 8:
        continue

    # Extracting data from the columns
    symbol = cols[0].text.strip()
    name = cols[1].text.strip()

    # Check the content of cols[3] for the price and change
    price_and_change = cols[3].text.strip()  # Price and change are here
    # print(f"Price and Change (raw): {price_and_change}")  # Debug print

    # Split the string to get price, change, and percentage change
    if price_and_change:
        price_change_split = price_and_change.split(' ')  # Split by space to separate price and change
        price = '$' + price_change_split[0]  # The first part is the price with $ sign
        change = price_change_split[1]  # The second part is the absolute change
        percent_change = price_change_split[2] if len(price_change_split) > 2 else None  # The third part is the % change
    else:
        price = None
        change = None
        percent_change = None

    # Extract market cap (no Volume column)
    marketcap = cols[8].text.strip()

    # Append the data to the list (excluding volume)
    data.append([symbol, name, price, change, percent_change, marketcap])

# Define column names (without Volume)
columns = ['Symbol', 'Name', 'Price', 'Change', '% Change', 'Market Cap']

# Create the DataFrame
df = pd.DataFrame(data, columns=columns)

# Display the DataFrame
# print(df)

# --- Email Functionality ---

# Retrieve email credentials from environment variables
sender_email = os.getenv("SENDER_EMAIL")  # Your email address
sender_password = os.getenv("SENDER_PASSWORD")  # Your app password or regular email password

# Function to send an email
def send_email(subject, body, recipient_email):
    # Create the email message
    message = MIMEMultipart()
    message['From'] = sender_email
    message['To'] = ", ".join(recipient_emails)
    message['Subject'] = subject

    message.attach(MIMEText(body, 'html'))

    # Send the email via SMTP
    try:
        with smtplib.SMTP('smtp.gmail.com', 587) as server:  # For Gmail
            server.starttls()  # Encrypt the connection
            server.login(sender_email, sender_password)
            server.sendmail(sender_email, recipient_email, message.as_string())
            print("Email sent successfully!")
    except Exception as e:
        print("Error sending email:", e)

# Convert the DataFrame to HTML
html_table = df.to_html(index=False)

# Email content
email_subject = "Top 7  Gainers and Undervalued Stocks Today"
email_body = f"""
<html>
  <body>
    <h2>Top 7 Gainers Stocks </h2>
    <p>Here are today's top 7 gainers:</p>
    {html_table}
    <br>
    <h2>Top 7 Undervalued Growth Stocks </h2>
    <p>Here are today's undervalued growth stocks:</p>
    {undervalued_html}
  </body>
</html>
"""
# Send the email
recipient_emails = [sender_email]
# recipient_email = sender_email  # You can send it to yourself or another email address
send_email(email_subject, email_body, recipient_emails)


  Symbol                      Name   Price Change % Change Market Cap
0     GM    General Motors Company  $51.37  -1.90   -3.57%    56.486B
1     ET        Energy Transfer LP  $19.71  +0.12   +0.61%    67.485B
2    APA           APA Corporation  $23.38  +0.29   +1.26%     8.649B
3    KGC  Kinross Gold Corporation   $9.88  +0.61   +6.58%    12.143B
4   LAUR  Laureate Education, Inc.  $18.15  -0.14   -0.77%     2.735B
5    BKR      Baker Hughes Company  $41.56  +0.54   +1.32%    41.125B
6   MTCH         Match Group, Inc.  $32.61  -0.10   -0.31%     8.188B
Email sent successfully!
